<a href="https://colab.research.google.com/github/Abhijith2005binu/AI-ML/blob/main/logistic_regression_gui_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q --upgrade gradio pandas numpy scikit-learn matplotlib seaborn

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully!")
print("Gradio version:", gr.__version__)

Libraries imported successfully!
Gradio version: 6.29.0


In [3]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

sample_df = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

sample_df["target"] = data.target

sample_df.to_csv(
    "breast_cancer.csv",
    index=False
)

print("Dataset created successfully!")
print("Shape:", sample_df.shape)

display(sample_df.head())

Dataset created successfully!
Shape: (569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [4]:
# Load dataset
df = pd.read_csv("breast_cancer.csv")

# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression
model = LogisticRegression(max_iter=1000)

model.fit(
    X_train_scaled,
    y_train
)

# Prediction
y_pred = model.predict(
    X_test_scaled
)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("LOGISTIC REGRESSION RESULTS")
print("=" * 40)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred
    )
)

LOGISTIC REGRESSION RESULTS
Accuracy  : 0.9825
Precision : 0.9861
Recall    : 0.9861
F1 Score  : 0.9861

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [5]:
# Global variables

df = None
model = None
scaler = None
label_encoder = None

feature_columns = []
target_column = None

X_test = None
y_test = None


def load_dataset(file):

    global df

    if file is None:
        return (
            "Please upload a CSV file.",
            gr.Dropdown(
                choices=[],
                value=None
            ),
            ""
        )

    try:

        # Gradio may return a filepath
        # or a file object depending on version

        if isinstance(file, str):
            file_path = file
        else:
            file_path = file.name

        df = pd.read_csv(file_path)

        columns = list(df.columns)

        preview = df.head(10).to_string(
            index=False
        )

        status = (
            "Dataset loaded successfully!\n\n"
            f"Rows: {df.shape[0]}\n"
            f"Columns: {df.shape[1]}"
        )

        return (
            status,
            gr.Dropdown(
                choices=columns,
                value=columns[-1]
            ),
            preview
        )

    except Exception as e:

        return (
            f"Error loading dataset:\n{str(e)}",
            gr.Dropdown(
                choices=[],
                value=None
            ),
            ""
        )


def train_model(target, test_size):

    global df
    global model
    global scaler
    global label_encoder
    global feature_columns
    global target_column
    global X_test
    global y_test

    if df is None:

        return (
            "Please upload a dataset first.",
            None
        )

    if target is None:

        return (
            "Please select a target column.",
            None
        )

    try:

        target_column = target

        # -------------------------------
        # Separate X and y
        # -------------------------------

        X = df.drop(
            columns=[target_column]
        )

        y = df[target_column]

        # -------------------------------
        # Keep numerical features
        # -------------------------------

        X = X.select_dtypes(
            include=np.number
        )

        if X.shape[1] == 0:

            return (
                "ERROR: No numerical feature columns found.",
                None
            )

        feature_columns = list(
            X.columns
        )

        # -------------------------------
        # Handle missing values
        # -------------------------------

        X = X.fillna(
            X.median(numeric_only=True)
        )

        # -------------------------------
        # Encode target
        # -------------------------------

        label_encoder = LabelEncoder()

        y = label_encoder.fit_transform(
            y.astype(str)
        )

        # -------------------------------
        # Check number of classes
        # -------------------------------

        if len(
            np.unique(y)
        ) < 2:

            return (
                "ERROR: Target column must contain at least two classes.",
                None
            )

        # -------------------------------
        # Train-test split
        # -------------------------------

        test_size = float(
            test_size
        )

        X_train, X_test_data, y_train, y_test_data = train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=42,
            stratify=y
        )

        X_test = X_test_data
        y_test = y_test_data

        # -------------------------------
        # Feature scaling
        # -------------------------------

        scaler = StandardScaler()

        X_train_scaled = scaler.fit_transform(
            X_train
        )

        X_test_scaled = scaler.transform(
            X_test_data
        )

        # -------------------------------
        # Logistic Regression
        # -------------------------------

        model = LogisticRegression(
            max_iter=1000
        )

        model.fit(
            X_train_scaled,
            y_train
        )

        # -------------------------------
        # Prediction
        # -------------------------------

        y_pred = model.predict(
            X_test_scaled
        )

        # -------------------------------
        # Metrics
        # -------------------------------

        accuracy = accuracy_score(
            y_test_data,
            y_pred
        )

        precision = precision_score(
            y_test_data,
            y_pred,
            average="weighted",
            zero_division=0
        )

        recall = recall_score(
            y_test_data,
            y_pred,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_test_data,
            y_pred,
            average="weighted",
            zero_division=0
        )

        # -------------------------------
        # Classification report
        # -------------------------------

        report = classification_report(
            y_test_data,
            y_pred,
            target_names=[
                str(c)
                for c in label_encoder.classes_
            ],
            zero_division=0
        )

        # -------------------------------
        # Confusion Matrix
        # -------------------------------

        cm = confusion_matrix(
            y_test_data,
            y_pred
        )

        fig, ax = plt.subplots(
            figsize=(6, 5)
        )

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_,
            ax=ax
        )

        ax.set_title(
            "Logistic Regression - Confusion Matrix"
        )

        ax.set_xlabel(
            "Predicted Class"
        )

        ax.set_ylabel(
            "Actual Class"
        )

        plt.tight_layout()

        # -------------------------------
        # Result
        # -------------------------------

        result = f"""
LOGISTIC REGRESSION RESULTS
============================================

Dataset
--------------------------------------------
Rows              : {df.shape[0]}
Columns           : {df.shape[1]}

Target Column     : {target_column}

Number of Features: {len(feature_columns)}

Training Samples  : {len(X_train)}
Testing Samples   : {len(X_test_data)}

Features
--------------------------------------------
{', '.join(feature_columns)}

MODEL PERFORMANCE
============================================

Accuracy          : {accuracy:.4f}
Precision         : {precision:.4f}
Recall            : {recall:.4f}
F1 Score          : {f1:.4f}

CLASSIFICATION REPORT
============================================

{report}
"""

        return (
            result,
            fig
        )

    except Exception as e:

        return (
            f"TRAINING ERROR:\n\n{type(e).__name__}: {str(e)}",
            None
        )

In [6]:
def predict_new_data(values):

    global model
    global scaler
    global feature_columns
    global label_encoder

    if model is None:

        return "Please train the model first."

    try:

        if len(values) != len(feature_columns):

            return (
                f"Expected {len(feature_columns)} "
                f"features, but received {len(values)}."
            )

        # Convert values to float
        values = [
            float(value)
            for value in values
        ]

        # Create DataFrame
        input_data = pd.DataFrame(
            [values],
            columns=feature_columns
        )

        # Scale
        input_scaled = scaler.transform(
            input_data
        )

        # Prediction
        prediction = model.predict(
            input_scaled
        )

        probabilities = model.predict_proba(
            input_scaled
        )[0]

        predicted_class = (
            label_encoder.inverse_transform(
                prediction
            )[0]
        )

        confidence = (
            np.max(probabilities) * 100
        )

        probability_text = ""

        for class_name, probability in zip(
            label_encoder.classes_,
            probabilities
        ):

            probability_text += (
                f"{class_name}: "
                f"{probability * 100:.2f}%\n"
            )

        result = f"""
PREDICTION RESULT
========================================

Predicted Class : {predicted_class}

Confidence      : {confidence:.2f}%

CLASS PROBABILITIES
----------------------------------------

{probability_text}
"""

        return result

    except ValueError:

        return (
            "ERROR: Please enter valid numerical values."
        )

    except Exception as e:

        return (
            f"Prediction Error:\n{str(e)}"
        )

In [7]:
with gr.Blocks(
    title="Logistic Regression Classifier"
) as app:

    gr.Markdown(
        """
        # Logistic Regression Classification

        ### Machine Learning Classification GUI

        Upload a CSV dataset → Select target → Train model →
        View performance → Predict new data
        """
    )

    # =====================================================
    # DATASET TAB
    # =====================================================

    with gr.Tab("Dataset"):

        file_input = gr.File(
            label="Upload CSV Dataset",
            file_types=[".csv"],
            type="filepath"
        )

        load_button = gr.Button(
            "Load Dataset",
            variant="primary"
        )

        dataset_status = gr.Textbox(
            label="Dataset Status",
            lines=4
        )

        target_dropdown = gr.Dropdown(
            label="Select Target Column",
            choices=[]
        )

        dataset_preview = gr.Textbox(
            label="Dataset Preview",
            lines=12
        )

        load_button.click(
            fn=load_dataset,
            inputs=file_input,
            outputs=[
                dataset_status,
                target_dropdown,
                dataset_preview
            ]
        )

    # =====================================================
    # TRAINING TAB
    # =====================================================

    with gr.Tab("Train Model"):

        test_size = gr.Dropdown(
            label="Testing Data Size",
            choices=[
                "0.20",
                "0.25",
                "0.30"
            ],
            value="0.20"
        )

        train_button = gr.Button(
            "Train Logistic Regression",
            variant="primary"
        )

        results_output = gr.Textbox(
            label="Model Results",
            lines=30
        )

        confusion_plot = gr.Plot(
            label="Confusion Matrix"
        )

        train_button.click(
            fn=train_model,
            inputs=[
                target_dropdown,
                test_size
            ],
            outputs=[
                results_output,
                confusion_plot
            ]
        )

    # =====================================================
    # PREDICTION TAB
    # =====================================================

    with gr.Tab("Prediction"):

        gr.Markdown(
            """
            ### Enter feature values

            This section is best used after training the model.
            """
        )

        feature_input = gr.Textbox(
            label="Feature Values",
            placeholder="Example: 10, 20.5, 30, 40",
            lines=2
        )

        predict_button = gr.Button(
            "Predict",
            variant="primary"
        )

        prediction_output = gr.Textbox(
            label="Prediction Result",
            lines=12
        )

        def prediction_from_text(text):

            try:

                values = [
                    float(x.strip())
                    for x in text.split(",")
                ]

                return predict_new_data(
                    values
                )

            except:

                return (
                    "Please enter numerical values "
                    "separated by commas."
                )

        predict_button.click(
            fn=prediction_from_text,
            inputs=feature_input,
            outputs=prediction_output
        )


app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://424c6b4446d621fd24.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
